In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### pipe构建

In [4]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.regression import *
from easymlops.table.ensemble import *

In [5]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
  .pipe(TargetEncoding(cols=["Name"],y=y_train))\
  .pipe(Parallel([
    LGBMRegression(y=y_train,objective="poisson",prefix="lgbm",drop_input_data=False),
    LogisticRegression(y=y_train,prefix="logist"),
    LinearRegression(y=y_train,prefix="lr"),
    RidgeRegression(y=y_train,prefix="ridge"),
    RidgeCVRegression(y=y_train,prefix="ridgecv")
]))

x_test_new=table.fit(x_train).transform(x_test)
x_test_new.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,...,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,lgbm_pred,logist_pred,lr_pred,ridge_pred,ridgecv_pred
500,501,0.482439,-1.0,1.111379,17.0,0,0,0.0,8.664062,0.299607,...,1,0,0,1,0,0.092996,0,0.129741,0.130264,0.134868
501,502,0.482439,-1.0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,...,1,0,0,0,1,0.716432,1,0.717859,0.717012,0.710683
502,503,0.482439,-1.0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,...,1,0,0,0,1,0.654662,1,0.779396,0.778454,0.771276
503,504,0.482439,-1.0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,...,1,0,0,0,1,0.309583,1,0.614467,0.614721,0.616731
504,505,-0.741789,-1.0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,...,0,1,0,0,1,0.916929,1,0.911410,0.911452,0.910562


In [6]:
# table=TablePipeLine()
# table.pipe(FixInput())\
#   .pipe(FillNa())\
#   .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
#   .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
#   .pipe(TargetEncoding(cols=["Name"],y=y_train))\
#   .pipe(Parallel([
#     LGBMRegression(y=y_train,objective="poisson",prefix="lgbm"),
#     LogisticRegression(y=y_train,prefix="logist"),
#     LinearRegression(y=y_train,prefix="lr"),
#     RidgeRegression(y=y_train,prefix="ridge"),
#     RidgeCVRegression(y=y_train,prefix="ridgecv")
# ]))

# x_test_new=table.fit(x_train).transform(x_test)
# x_test_new.head(5)

In [7]:
#获取sabass特征重要性
table[-1][0].get_contrib(table.transform_single(x_test.to_dict("record")[0],run_to_layer=-2))

{'Sex': -0.5899834278029109,
 'Cabin': -0.13150175316983173,
 'Fare': -0.1301026457360956,
 'Age': 0.275645992578784,
 'PassengerId': -0.39285142163510006,
 'Pclass_2': 0.0868239954973954,
 'SibSp': 0.048242598427820915,
 'Parch': -0.01823350433998343,
 'Pclass': -0.01402473665988247,
 'Ticket': -0.0030424253099006976,
 'Embarked': -0.005601108385329127}

### 性能、一致性测试

In [ ]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.4ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.WOEEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.TargetEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.ensemble.Parallel'>) module check [transform] complete,speed:[17.73ms]/it,cpu:[79%],memory:[116K]

###########################################

### 持久化测试

In [9]:
table.save("ml.pkl")

In [10]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
  .pipe(TargetEncoding(cols=["Name"],y=y_train))\
  .pipe(Parallel([
    LGBMRegression(objective="poisson",prefix="lgbm"),
    LogisticRegression(prefix="logist"),
    LinearRegression(prefix="lr"),
    RidgeRegression(prefix="ridge"),
    RidgeCVRegression(prefix="ridgecv")
]))

table.load("ml.pkl")

In [11]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.WOEEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.TargetEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.ensemble.Parallel'>) module check [transform] complete,speed:[17.87ms]/it,cpu:[60%],memory:[68K]

############################################

In [12]:
table.transform(x_test[:5])

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,...,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,lgbm_pred,logist_pred,lr_pred,ridge_pred,ridgecv_pred
500,501,0.482439,-1.0,1.111379,17.0,0,0,0.0,8.664062,0.299607,...,1,0,0,1,0,0.092996,0,0.129741,0.130264,0.134868
501,502,0.482439,-1.0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,...,1,0,0,0,1,0.716432,1,0.717859,0.717012,0.710683
502,503,0.482439,-1.0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,...,1,0,0,0,1,0.654662,1,0.779396,0.778454,0.771276
503,504,0.482439,-1.0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,...,1,0,0,0,1,0.309583,1,0.614467,0.614721,0.616731
504,505,-0.741789,-1.0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,...,0,1,0,0,1,0.916929,1,0.911410,0.911452,0.910562


In [13]:
x_test_new.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,...,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,lgbm_pred,logist_pred,lr_pred,ridge_pred,ridgecv_pred
500,501,0.482439,-1.0,1.111379,17.0,0,0,0.0,8.664062,0.299607,...,1,0,0,1,0,0.092996,0,0.129741,0.130264,0.134868
501,502,0.482439,-1.0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,...,1,0,0,0,1,0.716432,1,0.717859,0.717012,0.710683
502,503,0.482439,-1.0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,...,1,0,0,0,1,0.654662,1,0.779396,0.778454,0.771276
503,504,0.482439,-1.0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,...,1,0,0,0,1,0.309583,1,0.614467,0.614721,0.616731
504,505,-0.741789,-1.0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,...,0,1,0,0,1,0.916929,1,0.911410,0.911452,0.910562
